In [ ]:

# ============================================================
# WEEK 2: PROVIDER-AGNOSTIC LLM CLIENT
# Gemini API + Ollama (Llama 3.2)
# ============================================================

!pip install -q google-genai requests python-dotenv

import os
import requests
import subprocess
import time
import unittest

from abc import ABC, abstractmethod
from google import genai
from dotenv import load_dotenv

# ------------------------------------------------------------
# STEP 1: CONFIGURATION
# ------------------------------------------------------------

# For Gemini, add GEMINI_API_KEY to Colab Secrets.
# For Ollama, make sure Ollama is installed and Llama 3.2 is pulled.

# ------------------------------------------------------------
# STEP 2: COMMON LLM PROVIDER INTERFACE
# ------------------------------------------------------------

class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


# ------------------------------------------------------------
# STEP 3: GEMINI CLOUD PROVIDER
# ------------------------------------------------------------

class GeminiProvider(LLMProvider):

    def __init__(self, api_key):
        self.client = genai.Client(api_key=api_key)

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt
        )
        return response.text


# ------------------------------------------------------------
# STEP 4: OLLAMA LOCAL PROVIDER
# ------------------------------------------------------------

class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):

        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )

        response.raise_for_status()

        return response.json()["response"]


# ------------------------------------------------------------
# STEP 5: COMMON LLM CLIENT
# ------------------------------------------------------------

class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)


# ------------------------------------------------------------
# STEP 6: TESTING USING A FAKE PROVIDER
# ------------------------------------------------------------

class FakeProvider:

    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


class TestLLMClient(unittest.TestCase):

    def test_generate(self):

        provider = FakeProvider()
        client = LLMClient(provider)

        result = client.generate("Explain AI")

        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )


# Run unit test
suite = unittest.TestLoader().loadTestsFromTestCase(TestLLMClient)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)

# ------------------------------------------------------------
# STEP 7: SELECT LLM PROVIDER
# ------------------------------------------------------------

print("\n====================================")
print("       WEEK 2 LLM CLIENT")
print("====================================")

provider_name = input(
    "\nChoose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":

    # Read API key from Colab Secrets
    from google.colab import userdata

    api_key = userdata.get("GEMINI_API_KEY")

    if not api_key:
        raise ValueError(
            "Gemini API key not found. Add GEMINI_API_KEY "
            "to Google Colab Secrets."
        )

    provider = GeminiProvider(api_key)

elif provider_name == "ollama":

    # Start Ollama server if it is not already running
    try:
        requests.get("http://localhost:11434/api/tags", timeout=3)
        print("Ollama server is already running.")

    except requests.exceptions.RequestException:

        if not __import__("shutil").which("ollama"):
            raise RuntimeError(
                "Ollama is not installed in this Colab runtime. "
                "Install Ollama and pull llama3.2 first."
            )

        ollama_process = subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        time.sleep(5)

    # Check model availability
    tags = requests.get(
        "http://localhost:11434/api/tags",
        timeout=10
    ).json()

    models = [
        model["name"] for model in tags.get("models", [])
    ]

    if not any(model.startswith("llama3.2") for model in models):
        raise RuntimeError(
            "Llama 3.2 is not downloaded. Run !ollama pull llama3.2"
        )

    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider. Choose gemini or ollama.")

# ------------------------------------------------------------
# STEP 8: CREATE CLIENT AND GENERATE RESPONSE
# ------------------------------------------------------------

client = LLMClient(provider)

prompt = input("\nEnter your question: ")

answer = client.generate(prompt)

print("\n====================================")
print("LLM RESPONSE")
print("====================================")
print(answer)

print("\n====================================")
print("UNIT TEST STATUS")
print("====================================")

if test_result.wasSuccessful():
    print("All tests passed!")
else:
    print("Some tests failed.")

print("\nWeek 2 practical completed!")


test_generate (__main__.TestLLMClient.test_generate) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.004s

OK



       WEEK 2 LLM CLIENT

Choose provider (gemini/ollama): ollama


RuntimeError: Ollama is not installed in this Colab runtime. Install Ollama and pull llama3.2 first.

In [ ]:
# Install required dependency
!apt-get update -qq
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for the server
for i in range(30):
    try:
        response = requests.get(
            "http://localhost:11434/api/tags",
            timeout=3
        )
        if response.status_code == 200:
            print("Ollama server is running!")
            break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server failed to start.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 68 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (14.4 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/loc

In [4]:
!ollama pull llama3.2

In [5]:
!ollama run llama3.2 "Explain Artificial Intelligence in simple words."

Artificial Intelligence (AI) is a way to create machines that can think and
and learn like humans. Here's a simple explanation:

Imagine you have a computer that can:

1. Understand what you're telling it (like a person understanding a sentenc
sentence)
2. Make decisions based on that understanding
3. Improve its own performance over time (like getting better at a game)

That's basically what AI is: a computer system that can perform tasks that 
normally require human intelligence, such as:

* Recognizing images and voices
* Understanding natural language (like reading a book or having a conversat
conversation)
* Making predictions and decisions

AI is not just about making machines smarter, but also about using those ma
machines to solve problems and make our lives easier.

There are different types of AI, including:

* Narrow AI (also known as Weak AI): designed to perform a specific task, l
like image recognition or language translation
* General AI (also known as Strong AI): a hypo